# Домашнее задание № 5
Бинарные деревья, BST и куча. Python.

## Общие функции
Узел дерева хранит значение и ссылки на двух потомков. Массив задаёт дерево по уровням: дети индекса `i` — `2*i+1` и `2*i+2`.

In [1]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
        self.balance_factor = 0


def build_tree(arr, i=0):
    if i >= len(arr) or arr[i] is None:
        return None
    root = TreeNode(arr[i])
    root.left = build_tree(arr, 2 * i + 1)
    root.right = build_tree(arr, 2 * i + 2)
    return root

## 1. Полное бинарное дерево
Проверяем BFS. Все уровни, кроме последнего, должны быть заполнены, а последний — слева направо. Если после пустого места встретился узел, дерево не полное. **Время O(n), память O(w)**, где `w` — ширина дерева.

In [2]:
from collections import deque


def is_complete(root):
    queue = deque([root])
    empty_seen = False

    while queue:
        node = queue.popleft()
        if node is None:
            empty_seen = True
            continue
        if empty_seen:
            return False
        queue.append(node.left)
        queue.append(node.right)

    return True


assert is_complete(build_tree([1, 2, 3, 4, 5, 6]))
assert not is_complete(build_tree([1, 2, 3, 4, None, 6]))
assert is_complete(None)
print(is_complete(build_tree([1, 2, 3, 4, 5, 6])))

True


## 2. Объединение K отсортированных массивов
В мин-куче храним первые доступные элементы массивов. Достаём минимум и добавляем следующий элемент из того же массива. **Время O(N log K), память O(K)** без учёта ответа, где `N` — общее число элементов.

In [3]:
import heapq


def merge_k_arrays(arrays):
    heap = []
    result = []

    for i, arr in enumerate(arrays):
        if arr:
            heapq.heappush(heap, (arr[0], i, 0))

    while heap:
        value, i, j = heapq.heappop(heap)
        result.append(value)
        if j + 1 < len(arrays[i]):
            heapq.heappush(heap, (arrays[i][j + 1], i, j + 1))

    return result


assert merge_k_arrays([[1, 4], [2, 5], [3, 6]]) == [1, 2, 3, 4, 5, 6]
assert merge_k_arrays([[], [1, 1], [-2, 3]]) == [-2, 1, 1, 3]
assert merge_k_arrays([]) == []
print(merge_k_arrays([[1, 4], [2, 5], [3, 6]]))

[1, 2, 3, 4, 5, 6]


## 3. K-й наименьший / наибольший элемент
Держим в куче только `k` чисел. Для k-го наименьшего используем максимум из `k` меньших (в Python — отрицательные числа), для наибольшего — минимум из `k` больших. **Время O(n log k), память O(k)**.

In [4]:
def kth_smallest(nums, k):
    if not 1 <= k <= len(nums):
        return None
    heap = []
    for x in nums:
        heapq.heappush(heap, -x)
        if len(heap) > k:
            heapq.heappop(heap)
    return -heap[0]


def kth_largest(nums, k):
    if not 1 <= k <= len(nums):
        return None
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]


assert kth_smallest([7, 2, 5, 1, 9], 3) == 5
assert kth_largest([7, 2, 5, 1, 9], 2) == 7
assert kth_smallest([2, 2, 3], 2) == 2
assert kth_largest([], 1) is None
print(kth_smallest([7, 2, 5, 1, 9], 3), kth_largest([7, 2, 5, 1, 9], 2))

5 7


## 4. K-й элемент в BST
Обход слева–корень–справа даёт числа по возрастанию, справа–корень–слева — по убыванию. Считаем посещённые узлы до `k`. **Время O(h + k), память O(h)**, где `h` — высота дерева.

In [5]:
def kth_bst(root, k, largest=False):
    if k < 1:
        return None
    stack = []
    cur = root

    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.right if largest else cur.left

        cur = stack.pop()
        k -= 1
        if k == 0:
            return cur.val
        cur = cur.left if largest else cur.right

    return None


bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, None, None, 13])
assert kth_bst(bst, 3) == 4
assert kth_bst(bst, 2, largest=True) == 13
assert kth_bst(bst, 10) is None
print(kth_bst(bst, 3), kth_bst(bst, 2, largest=True))

4 13


## 5. Balance factor
Сначала рекурсивно считаем высоты детей, затем записываем `balance_factor = высота слева − высота справа`. Высота пустого поддерева — 0. **Время O(n), память O(h)**.

In [6]:
def fill_balance_factors(root):
    def height(node):
        if node is None:
            return 0
        left_h = height(node.left)
        right_h = height(node.right)
        node.balance_factor = left_h - right_h
        return 1 + max(left_h, right_h)

    height(root)
    return root


bst = fill_balance_factors(build_tree([8, 3, 10, 1, 6, None, 14]))
assert bst.balance_factor == 0
assert bst.right.balance_factor == -1
assert bst.left.balance_factor == 0
chain = fill_balance_factors(build_tree([3, 2, None, 1]))
assert chain.balance_factor == 2
print(bst.balance_factor, bst.right.balance_factor)

0 -1


## 6. Зеркальное дерево
В каждом узле меняем местами левого и правого ребёнка, затем делаем то же для потомков. Каждый узел посещаем один раз. **Время O(n), память O(h)** на рекурсию.

In [7]:
def mirror_tree(root):
    if root is None:
        return None
    root.left, root.right = root.right, root.left
    mirror_tree(root.left)
    mirror_tree(root.right)
    return root


tree = build_tree([4, 2, 7, 1, 3, 6, 9])
mirror_tree(tree)
assert (tree.left.val, tree.right.val) == (7, 2)
assert (tree.left.left.val, tree.right.right.val) == (9, 1)
mirror_tree(tree)
assert (tree.left.val, tree.right.val) == (2, 7)
assert mirror_tree(None) is None
print(tree.val, tree.left.val, tree.right.val)

4 2 7
